#### Polly speaking model using AWS.

In [1]:
import boto3
from botocore.exceptions import BotoCoreError, ClientError
from IPython.display import Audio, display
from pathlib import Path

In [2]:
def get_polly_client(region_name="us-east-1"):
    session = boto3.Session(region_name=region_name)
    return session.client("polly")

In [3]:
def polly_tts_to_file(
    text: str,
    out_path: str = "polly_output.mp3",
    voice_id: str = "Joanna",
    engine: str = "neural",          # It can also be "standard"
    output_format: str = "mp3",      # "mp3", "ogg_vorbis", "pcm"
    region_name: str = "us-east-1",
    use_ssml: bool = False,
) -> str:
    if not text or not text.strip():
        raise ValueError("Text is empty.")

    polly = get_polly_client(region_name=region_name)

    try:
        resp = polly.synthesize_speech(
            Text=text,
            TextType="ssml" if use_ssml else "text",
            VoiceId=voice_id,
            Engine=engine,
            OutputFormat=output_format,
        )

        audio_stream = resp["AudioStream"]
        out_path = str(Path(out_path).resolve())
        with open(out_path, "wb") as f:
            f.write(audio_stream.read())

        return out_path

    except (BotoCoreError, ClientError) as e:
        raise RuntimeError(f"AWS Polly synthesize_speech failed: {e}") from e


In [10]:
path = polly_tts_to_file("Hello everyone! This is AWS Polly speaking.", out_path="hello.mp3")
display(Audio(filename=path))